# Projeto 1: Logística Quantitativa Aplicada - Análise Completa

**Autor:** Felippe G S Ramos  
**Data:** 14 de Fevereiro de 2026  
**Objetivo:** Análise da gestão de estoques do restaurante Gulla's


## 1. Importar Bibliotecas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Configurar estilo dos gráficos
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

## 2. Definir Parâmetros do Modelo

In [ ]:
# Parâmetros da Demanda (semanal)
media_demanda_semanal = 68.86
desvio_padrao_demanda_semanal = 20.06

# Parâmetros de Custo
custo_pedido = 45.00  # Custo fixo por pedido (K)
custo_item = 25.00  # Custo por unidade do item (c)
custo_falta = 18.00  # Custo por unidade em falta (p)
taxa_manutencao_anual = 0.25  # Taxa de custo de manutenção (i)

# Parâmetros de Tempo
lead_time_semanas = 1  # Lead time em semanas (L)
semanas_por_ano = 52

# Cálculos Intermediários
custo_manutencao_semanal = (custo_item * taxa_manutencao_anual) / semanas_por_ano
demanda_anual_media = media_demanda_semanal * semanas_por_ano

print(f"Demanda Anual Média: {demanda_anual_media:.2f} unidades")
print(f"Custo de Manutenção Semanal: R$ {custo_manutencao_semanal:.4f}/unidade")

## 3. Calcular Política de Estoque (Q, ROP)

In [ ]:
# Lote Econômico de Compra (EOQ)
Q_otimo = np.sqrt((2 * custo_pedido * demanda_anual_media) / (custo_item * taxa_manutencao_anual))
Q_otimo = int(Q_otimo)

# Nível de Serviço e Estoque de Segurança
nivel_servico_desejado = 0.95
z_score = 1.645  # Para 95% de nível de serviço

estoque_seguranca = z_score * desvio_padrao_demanda_semanal * np.sqrt(lead_time_semanas)
estoque_seguranca = int(estoque_seguranca)

# Ponto de Reposição (ROP)
ponto_reposicao = int((media_demanda_semanal * lead_time_semanas) + estoque_seguranca)

print(f"Lote Econômico (Q*): {Q_otimo} unidades")
print(f"Estoque de Segurança: {estoque_seguranca} unidades")
print(f"Ponto de Reposição (ROP): {ponto_reposicao} unidades")

## 4. Simulação de Monte Carlo

In [ ]:
def simular_politica_estoque(Q, ROP, num_semanas, num_simulacoes, semente_base):
    np.random.seed(semente_base)
    
    resultados = []
    
    for sim in range(num_simulacoes):
        estoque_atual = Q + ROP
        custo_total = 0
        custo_pedidos = 0
        custo_manutencao = 0
        custo_falta = 0
        unidades_em_falta = 0
        pedidos_feitos = 0
        lead_time_restante = 0
        pedido_em_transito = False
        estoque_acumulado = 0
        
        for semana in range(num_semanas):
            # Chegada de pedido
            if pedido_em_transito:
                lead_time_restante -= 1
                if lead_time_restante <= 0:
                    estoque_atual += Q
                    pedido_em_transito = False
            
            # Demanda
            demanda = max(0, int(np.random.normal(media_demanda_semanal, desvio_padrao_demanda_semanal)))
            
            # Atender demanda
            if estoque_atual >= demanda:
                estoque_atual -= demanda
            else:
                unidades_em_falta += demanda - estoque_atual
                estoque_atual = 0
            
            # Custos
            custo_manutencao += estoque_atual * custo_manutencao_semanal
            custo_falta += (demanda - min(demanda, estoque_atual)) * custo_falta
            estoque_acumulado += estoque_atual
            
            # Novo pedido
            if not pedido_em_transito and estoque_atual <= ROP:
                custo_pedidos += custo_pedido
                pedidos_feitos += 1
                pedido_em_transito = True
                lead_time_restante = lead_time_semanas
        
        custo_total = custo_pedidos + custo_manutencao + custo_falta
        nivel_servico = 1 - (unidades_em_falta / (media_demanda_semanal * num_semanas))
        estoque_medio = estoque_acumulado / num_semanas
        
        resultados.append({
            "custo_total": custo_total,
            "custo_pedido": custo_pedidos,
            "custo_manutencao": custo_manutencao,
            "custo_falta": custo_falta,
            "nivel_servico": nivel_servico,
            "estoque_medio": estoque_medio,
            "pedidos_feitos": pedidos_feitos
        })
    
    return pd.DataFrame(resultados)

# Executar simulação
print("Executando simulação de Monte Carlo...")
df_resultados = simular_politica_estoque(Q_otimo, ponto_reposicao, 52, 1000, 42)
print("Simulação concluída!")

# Resumo estatístico
print("\nResumo Estatístico:")
print(df_resultados.describe())

## 5. Visualizar Resultados

In [ ]:
# Gráficos dos resultados
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Resultados da Simulação de Monte Carlo para a Política de Estoque (Q, ROP)", fontsize=16)

# 1. Distribuição do Custo Total
sns.histplot(df_resultados["custo_total"], kde=True, ax=axes[0, 0])
axes[0, 0].set_title("Distribuição do Custo Total Anual")
axes[0, 0].set_xlabel("Custo Total (R$)")

# 2. Distribuição do Nível de Serviço
sns.histplot(df_resultados["nivel_servico"], kde=False, ax=axes[0, 1])
axes[0, 1].set_title("Distribuição do Nível de Serviço")
axes[0, 1].set_xlabel("Nível de Serviço")

# 3. Composição Média de Custos
custos_medios = df_resultados[["custo_pedido", "custo_manutencao", "custo_falta"]].mean()
custos_medios.plot(kind='bar', ax=axes[0, 2], color=["steelblue", "darkorange", "red"])
axes[0, 2].set_title("Composição Média dos Custos")
axes[0, 2].set_ylabel("Custo (R$)")

# 4. Distribuição do Estoque Médio
sns.histplot(df_resultados["estoque_medio"], kde=True, ax=axes[1, 0])
axes[1, 0].set_title("Distribuição do Estoque Médio")
axes[1, 0].set_xlabel("Estoque Médio (unidades)")

# 5. Box Plot dos Componentes de Custo
sns.boxplot(data=df_resultados[["custo_pedido", "custo_manutencao", "custo_falta"]], ax=axes[1, 1])
axes[1, 1].set_title("Box Plot dos Componentes de Custo")
axes[1, 1].set_ylabel("Custo (R$)")

# Remover o sexto subplot
fig.delaxes(axes[1, 2])

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

print("Gráficos exibidos com sucesso!")

## 6. Análise de Cenários

In [ ]:
# Definir cenários
cenarios = {
    "Cenário 1: Base (Q=378, ROP=102, SL=95%)": (378, 102),
    "Cenário 2: SL 90% (Q=378, ROP=86)": (378, 86),
    "Cenário 3: SL 97.5% (Q=378, ROP=118)": (378, 118),
    "Cenário 4: Lote Reduzido (Q=250, ROP=102)": (250, 102),
    "Cenário 5: Lote Aumentado (Q=500, ROP=102)": (500, 102)
}

# Executar simulações para cada cenário
resumo_cenarios = []
for nome, (Q, ROP) in cenarios.items():
    print(f"Simulando {nome}...")
    df = simular_politica_estoque(Q, ROP, 52, 500, 42)
    
    resumo_cenarios.append({
        "Cenário": nome,
        "Custo Total Médio": df["custo_total"].mean(),
        "Custo Pedido Médio": df["custo_pedido"].mean(),
        "Custo Manutenção Médio": df["custo_manutencao"].mean(),
        "Custo Falta Médio": df["custo_falta"].mean(),
        "Nível de Serviço Médio": df["nivel_servico"].mean(),
        "Estoque Médio": df["estoque_medio"].mean(),
        "Pedidos Médios": df["pedidos_feitos"].mean()
    })

df_resumo = pd.DataFrame(resumo_cenarios)
print("\nResumo dos Cenários:")
print(df_resumo.to_string())

## 7. Decisão Recomendada

In [ ]:
# Encontrar o cenário com menor custo
idx_melhor = df_resumo["Custo Total Médio"].idxmin()
melhor_cenario = df_resumo.iloc[idx_melhor]

print("\n" + "="*80)
print("DECISÃO RECOMENDADA")
print("="*80)
print(f"\n{melhor_cenario['Cenário']}")
print(f"\nCusto Total Anual: R$ {melhor_cenario['Custo Total Médio']:.2f}")
print(f"Nível de Serviço: {melhor_cenario['Nível de Serviço Médio']:.2%}")
print(f"Estoque Médio: {melhor_cenario['Estoque Médio']:.2f} unidades")
print(f"Pedidos por Ano: {melhor_cenario['Pedidos Médios']:.1f}")
print("\nEsta política minimiza os custos logísticos e praticamente elimina as faltas de estoque.")
print("="*80)